# Session 7 — Demo 4 — The finale: from a pile of texts to a report

**What this demo covers.** In week one you saw an HTML report built from the warehouse data and heard "by the end you'll build one". Here it is, for the complaints of Demo 3: real volumes with the January spike called out, eight themes with names and sizes, a map of all complaints you can hover over, and customers' own words. Everything in it was computed in this course: counts and charts (sessions 1 to 3), an honest look at a spike (session 3, and Demo 3 tonight), clusters and a map (tonight). An assistant wrote the HTML around our numbers; we checked what goes in.

**Data.** `data/complaints/`, as in Demo 3, plus the precomputed vectors from the same encoder, so this notebook runs in seconds without downloading the model.

**How to run.** Locally: top to bottom, then open `complaints-report.html` in a browser. Colab: uncomment the setup lines; the report appears in the Colab file panel, download it to open.

## 0. Setup

In [1]:
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import umap

DATA = Path("../../data")

# --- Google Colab? Uncomment these lines to fetch the course repo: ---
# !git clone https://github.com/seemsGoodNow/hse-data-science-course.git course
# DATA = Path("course/data")
# %pip install umap-learn

PRODUCT_SHORT = {
    "Checking or savings account": "Bank account",
    "Credit card": "Credit card",
    "Debt collection": "Debt collection",
    "Money transfer, virtual currency, or money service": "Money transfer",
    "Mortgage": "Mortgage",
    "Student loan": "Student loan",
    "Vehicle loan or lease": "Car loan",
}
CLUSTER_NAMES = {
    0: "Late payments and fees",
    1: "Card fraud and disputes",
    2: "Credit reports and debt collection",
    3: "Bank accounts and transfers",
    4: "Student loans",
    5: "Car loans",
    6: "Cash App and Zelle",
    7: "Mortgages and escrow",
}
CLUSTER_COLORS = [
    "#4a7ebb",
    "#c0392b",
    "#27ae60",
    "#8e44ad",
    "#e67e22",
    "#16a085",
    "#e84393",
    "#2c3e50",
]

## 1. The numbers that go into the report

Demo 3's pipeline, compressed: the same 5,000 complaints, the same encoder (its vectors are precomputed in the course data), the same k-means with the same names, and a UMAP map for the cover.

In [2]:
complaints = pd.read_parquet(DATA / "complaints" / "complaints.parquet")
monthly = pd.read_parquet(DATA / "complaints" / "monthly_counts.parquet")
precomputed = pd.read_parquet(DATA / "complaints" / "embeddings_minilm.parquet")

complaint_texts = complaints["narrative"].sample(5000, random_state=42)
sample_ids = complaints.loc[complaint_texts.index, "complaint_id"]
# each stored vector sits in one cell; np.vstack stacks them into one table
stored_vectors = precomputed.set_index("complaint_id").loc[sample_ids, "embedding"]
embeddings = np.vstack(stored_vectors)

complaint_kmeans = KMeans(n_clusters=8, n_init=10, random_state=42).fit(embeddings)
cluster_of = pd.Series(complaint_kmeans.labels_, index=complaint_texts.index)
theme_of = cluster_of.map(CLUSTER_NAMES)

# n_jobs=1: one processor core, which UMAP needs for a repeatable result
map_umap = umap.UMAP(
    n_components=2,
    random_state=42,
    n_jobs=1,
).fit_transform(embeddings)

theme_sizes = theme_of.value_counts()
theme_sizes

Bank accounts and transfers           987
Card fraud and disputes               730
Credit reports and debt collection    727
Mortgages and escrow                  642
Late payments and fees                612
Student loans                         605
Car loans                             509
Cash App and Zelle                    188
Name: count, dtype: int64

In [3]:
monthly["product_short"] = monthly["product"].map(PRODUCT_SHORT)
monthly_by_product = monthly.pivot_table(
    index="month",
    columns="product_short",
    values="total_complaints",
)

is_2025 = monthly_by_product.index >= "2025-01"
is_january = monthly_by_product.index == "2025-01"
average_2024 = monthly_by_product[~is_2025].mean()
average_2025_no_january = monthly_by_product[is_2025 & ~is_january].mean()
growth_without_january = average_2025_no_january / average_2024 - 1

total_complaints = int(monthly["total_complaints"].sum())
january_total = int(monthly_by_product[is_january].sum(axis=1).iloc[0])
usual_month = int(monthly_by_product[~is_january].sum(axis=1).median())

print(
    f"complaints in the database, 12 months: {total_complaints:,}\n"
    f"January 2025: {january_total:,}, a usual month: {usual_month:,}"
)
(growth_without_january * 100).round(0).sort_values(ascending=False)

complaints in the database, 12 months: 257,567
January 2025: 72,418, a usual month: 15,795


product_short
Money transfer     188.0
Student loan       101.0
Debt collection     37.0
Car loan            33.0
Bank account        24.0
Credit card         15.0
Mortgage             7.0
dtype: float64

Two quotes per theme for the biggest three themes: the texts closest to each theme's center, cut to 260 characters. `escape` makes a text safe to put inside HTML (a `<` in a complaint must not break the page).

In [4]:
distance_to_center = pd.DataFrame(complaint_kmeans.transform(embeddings))

quotes = {}
for cluster, theme in CLUSTER_NAMES.items():
    closest = distance_to_center[cluster].sort_values().head(2).index
    quotes[theme] = [
        escape(complaint_texts.iloc[position][:260].replace("\n", " ")) + "..."
        for position in closest
    ]
top_themes = list(theme_sizes.index[:3])
top_themes

['Bank accounts and transfers',
 'Card fraud and disputes',
 'Credit reports and debt collection']

`[... for position in closest]` builds a list in one line: one quote for every position in `closest`. It is a short form of a `for` loop that appends to a list.

## 2. The prompt

> You are a data analyst. Build a self-contained HTML report, "What US bank customers complained about", for a retail-bank executive. Inputs: monthly complaint counts by product (`monthly_by_product`), total complaints, the January 2025 total and a usual month, growth without January by product, eight themes with sizes (`theme_sizes`), 2D map coordinates with a theme per complaint (`map_umap`, `theme_of`), two quotes per theme (`quotes`) and the three biggest themes. Requirements: a header and four KPI cards; a monthly stacked chart with the January spike annotated; a horizontal bar chart of themes; an interactive map where hovering shows the start of the complaint; quotes for the top three themes as pull-quotes; a short "how this was made" note with the caveats (sampled texts, map not evidence). Plotly, calm colours, one file `complaints-report.html`. Return only the code.

## 3. The code the assistant returned

Don't read every line of styling. **Read what it computes and check three things:** where each KPI number comes from (our variables above, not new calculations), which texts become quotes (the `quotes` we built), and which file is written.

In [5]:
# --- code returned by the assistant: check the inputs, skim the styling ---

# chart 1: monthly volumes, stacked by product, with the January spike annotated
trend = go.Figure()
for column_number, product in enumerate(monthly_by_product.columns):
    trend.add_trace(go.Bar(
        x=monthly_by_product.index,
        y=monthly_by_product[product],
        name=product,
        marker_color=CLUSTER_COLORS[column_number],
    ))
trend.add_annotation(
    x="2025-01",
    y=january_total,
    text=f"January 2025: {january_total:,}<br>a one-month spike, not a trend",
    showarrow=True,
    arrowhead=2,
    ax=150,
    ay=-5,
    xanchor="left",
)
trend.update_layout(
    barmode="stack",
    template="plotly_white",
    height=380,
    margin=dict(l=50, r=20, t=20, b=40),
    legend=dict(orientation="h", y=-0.15),
    yaxis_title="complaints per month",
)

# chart 2: themes by size
theme_bars = go.Figure(go.Bar(
    x=theme_sizes.values[::-1],
    y=theme_sizes.index[::-1],
    orientation="h",
    marker_color="#4a7ebb",
    text=[f"{share:.0%}" for share in (theme_sizes / theme_sizes.sum())[::-1]],
    textposition="outside",
))
theme_bars.update_layout(
    template="plotly_white",
    height=360,
    margin=dict(l=20, r=40, t=20, b=40),
    xaxis_title="complaints in the sample of 5,000",
)

# chart 3: the map, one trace per theme, the start of each complaint on hover
complaint_map = go.Figure()
for cluster, theme in CLUSTER_NAMES.items():
    in_theme = (theme_of == theme).to_numpy()
    hover_texts = [
        escape(text[:140].replace("\n", " ")) + "..."
        for text in complaint_texts[in_theme]
    ]
    complaint_map.add_trace(go.Scattergl(
        x=map_umap[in_theme, 0],
        y=map_umap[in_theme, 1],
        mode="markers",
        name=theme,
        marker=dict(size=4, color=CLUSTER_COLORS[cluster], opacity=0.7),
        text=hover_texts,
        hovertemplate="%{text}<extra>" + theme + "</extra>",
    ))
# zoom to where 99% of the dots are, so a few far-away dots don't shrink the map
x_range = np.quantile(map_umap[:, 0], [0.005, 0.995])
y_range = np.quantile(map_umap[:, 1], [0.005, 0.995])
complaint_map.update_layout(
    template="plotly_white",
    height=560,
    margin=dict(l=10, r=10, t=10, b=10),
    xaxis=dict(visible=False, range=[x_range[0] - 0.5, x_range[1] + 0.5]),
    yaxis=dict(visible=False, range=[y_range[0] - 0.5, y_range[1] + 0.5]),
    legend=dict(orientation="h", y=-0.05, itemsizing="constant"),
)

fastest_product = growth_without_january.idxmax()
kpis = [
    (f"{total_complaints:,}", "complaints in the database, June 2024 to May 2025"),
    (f"{january_total / usual_month:.1f}×", "January 2025 against a usual month"),
    (
        f"+{growth_without_january[fastest_product]:.0%}",
        f"{fastest_product}, 2025 vs 2024, January excluded",
    ),
    (str(len(CLUSTER_NAMES)), "themes found in the texts, none given in advance"),
]
kpi_cards = "".join(
    f'<div class="kpi"><div class="value">{value}</div>'
    f'<div class="label">{label}</div></div>'
    for value, label in kpis
)
quote_blocks = "".join(
    f'<div class="theme"><h3>{theme}</h3>'
    + "".join(f"<blockquote>{quote}</blockquote>" for quote in quotes[theme])
    + "</div>"
    for theme in top_themes
)

page = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>What US bank customers complained about</title>
<style>
  body {{ margin: 0; background: #f6f5f1; color: #1a1a18;
         font-family: "Helvetica Neue", Arial, sans-serif; }}
  header {{ background: #1f2a44; color: #fff; padding: 48px 8vw 40px; }}
  header h1 {{ margin: 0 0 8px; font-size: 34px; font-weight: 600; }}
  header p {{ margin: 0; color: #c9d2e6; font-size: 17px; max-width: 760px; }}
  main {{ max-width: 1100px; margin: -28px auto 60px; padding: 0 24px; }}
  .kpis {{ display: grid; gap: 14px;
           grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); }}
  .kpi {{ background: #fff; border-radius: 12px; padding: 18px 20px;
          box-shadow: 0 2px 10px rgba(20, 30, 60, .08); }}
  .kpi .value {{ font-size: 30px; font-weight: 700; color: #1f2a44; }}
  .kpi .label {{ color: #6b6a66; font-size: 14px; margin-top: 4px; }}
  section {{ background: #fff; border-radius: 12px; padding: 22px 26px;
             margin-top: 22px; box-shadow: 0 2px 10px rgba(20, 30, 60, .06); }}
  section h2 {{ margin: 0 0 4px; font-size: 22px; }}
  section .lead {{ margin: 0 0 12px; color: #6b6a66; }}
  .themes {{ display: grid; gap: 18px;
             grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); }}
  .theme h3 {{ margin: 6px 0; font-size: 17px; color: #1f2a44; }}
  blockquote {{ margin: 10px 0; padding: 8px 14px; border-left: 3px solid #c15f3c;
                color: #444; font-style: italic; font-size: 14px;
                background: #faf8f4; }}
  footer {{ color: #6b6a66; font-size: 13px; max-width: 1100px;
            margin: 0 auto 40px; padding: 0 24px; }}
</style></head>
<body>
<header>
  <h1>What US bank customers complained about</h1>
  <p>A year of complaints to the US Consumer Financial Protection Bureau, read by a
    machine and checked by a person.</p>
</header>
<main>
  <div class="kpis">{kpi_cards}</div>
  <section>
    <h2>Monthly volumes</h2>
    <p class="lead">Money transfers spiked in January 2025 and settled at about twice
      their 2024 level; student loans doubled; debt collection grew steadily. January
      is an outlier and is excluded from growth figures.</p>
    {trend.to_html(full_html=False, include_plotlyjs=True)}
  </section>
  <section>
    <h2>What the complaints are about</h2>
    <p class="lead">Eight themes found in 5,000 complaint texts. Two of them cut
      across products: late fees and card fraud.</p>
    {theme_bars.to_html(full_html=False, include_plotlyjs=False)}
  </section>
  <section>
    <h2>A map of all complaints</h2>
    <p class="lead">Every dot is a complaint; similar complaints sit close
      together. Hover to read. The map is for orientation, not evidence.</p>
    {complaint_map.to_html(full_html=False, include_plotlyjs=False)}
  </section>
  <section>
    <h2>In their own words</h2>
    <p class="lead">The two complaints closest to the center of each of the three
      biggest themes, lightly cut.</p>
    <div class="themes">{quote_blocks}</div>
  </section>
</main>
<footer>
  How this was made: public CFPB data. Volumes come from full-database counts;
  the 5,000 texts are a sample from every month, not in proportion to volume, so
  they describe topics, not volumes. Themes: a sentence encoder (all-MiniLM-L6-v2) and k-means with
  k = 8, named by a person. Map: UMAP of the same vectors; sizes and distances
  on it are not exact. Personal details were masked by the regulator (XXXX).
</footer>
</body></html>"""

Path("complaints-report.html").write_text(page, encoding="utf-8")
report_megabytes = Path("complaints-report.html").stat().st_size / 1e6
print(f"complaints-report.html written: {report_megabytes:.1f} MB")

complaints-report.html written: 5.7 MB


Open `complaints-report.html` in a browser. Hover over the map, read the quotes, look at the January bar.

**In week one this looked like magic. Tonight every step has a name:** loading and counting (sessions 1 and 2), an honest chart and a spike you refused to call a trend (session 3), clusters, names and a map (tonight), and an assistant doing the HTML boilerplate around numbers you computed and checked. That is the whole course in one file.

## Try it yourself — the graduation exercise

1. Add a fifth KPI of your own (for example, the share of complaints in the biggest theme). One prompt, then check the number against `theme_sizes`.
2. Build the same report for **mortgage complaints only**: filter `complaints` first, re-cluster with fewer themes.
3. Point this pipeline at texts your future employer already has: support tickets, survey comments, meeting notes. That's the course, in your hands.

In [6]:
# your turn

## The course, in one recap

- Data is the trace of a process: understand the process, understand the table.
- A few verbs and joins move any table; count your rows after every join.
- Distribution first, number second; one month can make a "trend".
- Luck looks like signal: check it before you believe it.
- Models: split first, pick the metric that matches the money, a threshold is a business decision, too good to be true usually is.
- No answer column? Clustering: numbers or text in, groups you can name out; compress with PCA or UMAP when there are too many columns.
- And underneath all of it: **think, ask well, check.**

*Thank you for the seven Wednesdays.*